# Module 3 – Train, Validation & Test

## Dataset – Hotel Bookings

In this notebook, we will understand how to divide data into training, validation, and test datasets.

We will also learn:

- Train/Test Split
- Train/Validation/Test Split
- Cross Validation
- K-Fold Cross Validation
- Stratified K-Fold
- Random State
- Data Leakage
- Overfitting
- Underfitting

The target variable for this project is `is_canceled`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/hotel_bookings.csv")

df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [2]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (119390, 32)

Columns:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']

Missing Values:
hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                

## Preparing the Dataset

For this notebook, `is_canceled` is the target variable.

- `0` → Booking was not canceled
- `1` → Booking was canceled

We will use numerical features that are available before the booking outcome.

In [3]:
# Fill missing values in children
df["children"] = df["children"].fillna(0)

features = [
    "lead_time",
    "adults",
    "children",
    "babies",
    "adr",
    "total_of_special_requests"
]

X = df[features]
y = df["is_canceled"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   lead_time  adults  children  babies   adr  total_of_special_requests
0        342       2       0.0       0   0.0                          0
1        737       2       0.0       0   0.0                          0
2          7       1       0.0       0  75.0                          0
3         13       1       0.0       0  75.0                          0
4         14       2       0.0       0  98.0                          1

Target:
0    0
1    0
2    0
3    0
4    0
Name: is_canceled, dtype: int64


## 1. Training Dataset

The Training Dataset is the portion of data used to train the machine learning model.

The model learns patterns and relationships between the features and the target from the training data.

It is similar to a student learning from study materials before taking an exam.

## 2. Validation Dataset

The Validation Dataset is used during model development.

It helps us:

- Compare different models
- Tune hyperparameters
- Check model performance
- Detect overfitting

The validation dataset should not be treated as the final test data.

## 3. Test Dataset

The Test Dataset is kept separate and is used for the final evaluation of the model.

It represents unseen data and gives us an estimate of how the final model may perform on new data.

The test dataset should not be used for model tuning.

## 4. Train/Test Split

Train/Test Split divides the dataset into two parts:

- Training Data → Used to train the model
- Test Data → Used for final evaluation

A common split is 80% training data and 20% test data.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training size:", len(X_train))
print("Test size:", len(X_test))

Training size: 95512
Test size: 23878


## 5. Train/Validation/Test Split

For more controlled model development, we can divide the dataset into three parts:

- Training → 60%
- Validation → 20%
- Test → 20%

The training data is used to train the model.

The validation data is used to compare models and tune parameters.

The test data is used only for the final evaluation.

In [5]:
# First split: 80% temporary data and 20% test data
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: 75% training and 25% validation from temporary data
# This results in 60% train, 20% validation and 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Training size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

Training size: 71634
Validation size: 23878
Test size: 23878


## 6. Cross Validation

Cross Validation is a technique used to evaluate a machine learning model using multiple training and validation splits.

Instead of depending on only one validation split, the data is divided into multiple parts and the model is evaluated several times.

This provides a more reliable estimate of model performance.

## 7. K-Fold Cross Validation

K-Fold Cross Validation divides the data into K equal parts called folds.

For each round:

- One fold is used for validation.
- The remaining folds are used for training.
- This process is repeated until every fold has been used for validation.

The final score is calculated by taking the average of all fold scores.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score

model = LogisticRegression(max_iter=1000)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring="accuracy"
)

print("Scores for each fold:", scores)
print("Average score:", scores.mean())

Scores for each fold: [0.69272971 0.69164084 0.69557752 0.69272971 0.69729458]
Average score: 0.6939944718988189


## 8. Stratified K-Fold

Stratified K-Fold is mainly used for classification problems.

It maintains approximately the same proportion of each class in every fold.

Since `is_canceled` is a classification target, Stratified K-Fold is useful for this dataset.

In [7]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print("Scores for each fold:", scores)
print("Average score:", scores.mean())

Scores for each fold: [0.69281347 0.6915152  0.69239467 0.69620571 0.69712706]
Average score: 0.6940112237205797


## Checking Class Distribution in Stratified K-Fold

Stratified K-Fold keeps the distribution of the target classes similar across the different folds.

In [8]:
for fold, (train_index, val_index) in enumerate(skf.split(X, y), start=1):
    train_ratio = y.iloc[train_index].mean()
    val_ratio = y.iloc[val_index].mean()

    print(f"Fold {fold}")
    print(f"Training cancellation ratio: {train_ratio:.3f}")
    print(f"Validation cancellation ratio: {val_ratio:.3f}")
    print()

Fold 1
Training cancellation ratio: 0.370
Validation cancellation ratio: 0.370

Fold 2
Training cancellation ratio: 0.370
Validation cancellation ratio: 0.370

Fold 3
Training cancellation ratio: 0.370
Validation cancellation ratio: 0.370

Fold 4
Training cancellation ratio: 0.370
Validation cancellation ratio: 0.370

Fold 5
Training cancellation ratio: 0.370
Validation cancellation ratio: 0.370



## 9. Random State

`random_state` is used to control randomness during operations such as data splitting and shuffling.

Using the same random state gives the same result every time.

This makes our experiments reproducible and easier to compare.

In [9]:
split_a = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

split_b = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Same training split:", split_a[0].equals(split_b[0]))

Same training split: True


## 10. Data Leakage

Data Leakage happens when information that should not be available during model training is used by the model.

This can make the model appear to perform better than it actually will on new data.

### Examples:

- Using test data during model training
- Scaling the complete dataset before splitting
- Using information that becomes available only after the prediction event
- Duplicate records appearing in both training and test datasets

For the Hotel Bookings dataset, columns such as `reservation_status` and `reservation_status_date` can cause leakage when predicting cancellation before the booking outcome is known.

### Rule:

Split the data first, then fit preprocessing steps only on the training data.

## 11. Overfitting

Overfitting happens when a model learns the training data too closely, including noise and random patterns.

An overfitted model usually performs very well on training data but performs worse on unseen validation or test data.

### Example:

A student memorizes the answers from practice questions but cannot solve new questions.

## 12. Underfitting

Underfitting happens when a model is too simple to learn the important patterns in the data.

An underfitted model performs poorly on both training data and unseen data.

### Example:

A student does not learn enough concepts and performs poorly in both practice tests and the final exam.

## 13. Why Should the Test Dataset Not Be Used for Model Tuning?

The test dataset should remain completely separate during model development.

We use the training data to train the model and validation data or cross validation to compare models and tune hyperparameters.

The test dataset is used only after the final model has been selected.

If we repeatedly use the test dataset for tuning, our decisions can become influenced by the test results. This means the test dataset is no longer truly unseen.

### Correct Workflow

Training Data
↓
Model Training
↓
Validation / Cross Validation
↓
Model & Hyperparameter Selection
↓
Final Model
↓
Test Data
↓
Final Evaluation

The test dataset should therefore be used only once for the final evaluation.

## Conclusion

In this notebook, we learned how to divide data into training, validation, and test datasets.

We also learned:

- Train/Test Split
- Train/Validation/Test Split
- Cross Validation
- K-Fold Cross Validation
- Stratified K-Fold
- Random State
- Data Leakage
- Overfitting
- Underfitting

Keeping the test dataset separate is important for getting a reliable estimate of final model performance.